# Full Triple Matcher with Semantic Type Matching, Adaptive Threshold Optimization, and Conflict Resolution

## B.Tech CSE Final Year Research Project — Comprehensive Defense Benchmark
**Base Paper:** *"Full Triple Matcher: Integrating All Triple Elements between Heterogeneous Knowledge Graphs"*  
Victor Eiti Yamamoto and Hideaki Takeda, **ACM Transactions on the Web (TWEB)**, 2025.  

---

### Research Contributions & Innovations:
This research introduces three architectural enhancements to the foundational Full Triple Matcher (FTM) framework:
1. **Improvement 1 — Semantic Type-Aware Matching:** Extracts ontological categories and structural domain roles to compute semantic compatibility bonuses and prevent cross-type alignment errors (e.g. matching an Actor to a Director or Film to a City).
2. **Improvement 2 — Adaptive Threshold Optimization:** Replaces arbitrary global thresholds (e.g. fixed $\theta = 0.80$) with a candidate-dependent dynamic cut-off formulated as:
   $$\theta_{\text{adaptive}} = \theta_{\text{base}} + \lambda_{\text{unc}} \cdot \text{Penalty} - \lambda_{\text{conf}} \cdot \text{Bonus}_{\text{sim}} - \lambda_{\text{sem}} \cdot \text{Bonus}_{\text{sem}}$$
3. **Improvement 3 — Automated Conflict Resolution:** Implements multi-stage arbitration over 1-to-many, many-to-1, and competing candidate conflicts utilizing multi-source evidence (surface similarity, triple similarity, candidate margin, and semantic compatibility).

---

### End-to-End Agnostic System Pipeline:
$$\text{User Selected Dataset / KG Pair} \longrightarrow \text{Candidate Generation} \longrightarrow \mathbf{\text{Baseline FTM}} \longrightarrow \mathbf{\text{Imp 1 (Semantic Type)}} \longrightarrow \mathbf{\text{Imp 2 (Adaptive Threshold)}} \longrightarrow \mathbf{\text{Imp 3 (Conflict Resolution)}} \longrightarrow \text{Final Alignment & Evaluation}$$

In [ ]:
# Section 1: System Initialization & Dynamic Module Loading
import os
import sys
import json
import pandas as pd
import numpy as np

# Locate project root dynamically
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
curr = os.path.abspath(notebook_dir)
while curr and os.path.dirname(curr) != curr:
    if os.path.isdir(os.path.join(curr, "codes")) and os.path.isdir(os.path.join(curr, "dataset")):
        break
    curr = os.path.dirname(curr)
project_root = curr

entity_matching_dir = os.path.join(project_root, "codes", "entity-matching")
config_dir = os.path.join(project_root, "codes", "config")
pipeline_dir = os.path.join(project_root, "codes", "pipeline")
utils_dir = os.path.join(project_root, "codes", "utils")

for p in [project_root, entity_matching_dir, config_dir, pipeline_dir, utils_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

from file_utils import load_csv, load_json, load_rdf_graph
from gpu_utils import get_gpu_info, is_cuda_available
import semantic_type_matching as stm
import adaptive_threshold as at
import conflict_resolution as cr
from entity_matching import run_pipeline, EntityMatcherPipeline
from evaluation import evaluate_predictions, build_comparison_dataframe
from dataset_config import resolve_dataset_config, validate_dataset_config, list_available_datasets, DATASETS

print("=" * 65)
print("FULL-TRIPLE-MATCHER (FTM) RESEARCH ENVIRONMENT")
print("=" * 65)
print(f"Project root: {project_root}")
gpu_info = get_gpu_info()
print(f"CUDA Available: {gpu_info['cuda_available']} | Devices: {gpu_info['device_count']}")
print("Available Benchmarks:")
for d in list_available_datasets(project_root_dir=project_root):
    print(f"  * {d['name']:<12} : {d['display_name']}")

## Section 2 — Configuration

Select any benchmark dataset (`movies`, `newmovies`, or `dw-nb`), or explicitly specify custom Knowledge Graph files below.
The matching algorithms are **completely dataset/KG-agnostic** and do not assume fixed filenames or domains.

In [ ]:
# ==============================================================================
# BENCHMARK / KG CONFIGURATION
# ==============================================================================
# Select a dataset: 'movies', 'newmovies', or 'dw-nb'
DATASET_NAME = "movies"

RUN_ALL_MODES = True
USE_GPU = True
OUTPUT_ROOT = "results"

# Optional: Set explicit KG paths to bypass named datasets entirely
# (Leave as None to use the selected DATASET_NAME above)
KG1_PATH = None
KG2_PATH = None
CANDIDATES_PATH = None
GOLD_PATH = None

## Section 3 — Display Selected Inputs & Dataset Validation

Verify and validate all input Knowledge Graph and alignment files before running algorithms.

In [ ]:
# Resolve and validate configuration
config = resolve_dataset_config(
    dataset_name=DATASET_NAME,
    kg1=KG1_PATH,
    kg2=KG2_PATH,
    candidates=CANDIDATES_PATH,
    gold=GOLD_PATH,
    output_dir=OUTPUT_ROOT
)

validation_result = validate_dataset_config(config, require_candidates=False, require_gold=False, verbose=False)

print("=" * 65)
print("SELECTED PIPELINE INPUTS")
print("=" * 65)
print(f"Selected Dataset: {config.get('display_name')}")
print(f"Dataset Identifier: {config.get('dataset_name')}")
print(f"KG1 Path:         {config.get('kg1')}")
print(f"KG2 Path:         {config.get('kg2')}")
print(f"Candidates Path:  {config.get('candidates') or '(Dynamic graph candidate generation)'}")
print(f"Gold Standard:    {config.get('gold') or '(Evaluation skipped - no gold file)'}")
print(f"Output Directory: {config.get('output_dir') or OUTPUT_ROOT}")
print(f"GPU Requested:    {USE_GPU} (CUDA Available: {gpu_info['cuda_available']})")
print(f"Validation:       {'PASSED [All paths verified]' if validation_result['valid'] else 'FAILED'}")
print("=" * 65)

## Section 4 — Load Inputs Generically

Data loading is entirely generic and driven by the resolved configuration object without any hard-coded file assumptions.

In [ ]:
# Load candidate pairs
if config.get("candidates") and os.path.exists(config["candidates"]):
    candidates_df = load_csv(config["candidates"])
    print(f"[Candidates] Successfully loaded {len(candidates_df):,} candidate pairs:")
    display(candidates_df.head(5))
else:
    candidates_df = None
    print("[Candidates] No precomputed CSV provided; candidates will be extracted dynamically.")

# Load gold standard ground truth
if config.get("gold") and os.path.exists(config["gold"]):
    gold_df = load_csv(config["gold"])
    print(f"\n[Gold Standard] Successfully loaded {len(gold_df):,} ground truth alignments:")
    display(gold_df.head(5))
else:
    gold_df = None
    print("\n[Gold Standard] No gold standard provided (inference-only mode).")

# Load Knowledge Graphs (if graph parsing is configured)
if config.get("load_graphs", True) and config.get("kg1") and os.path.exists(config["kg1"]):
    graph1 = load_rdf_graph(config["kg1"])
    graph2 = load_rdf_graph(config["kg2"])
    print(f"\n[RDF Graphs] KG1: {len(graph1):,} triples | KG2: {len(graph2):,} triples")
else:
    graph1, graph2 = None, None
    print("\n[RDF Graphs] Graph parsing skipped (large benchmark mode with precomputed similarities).")

## Section 5 — Run The Four-Way Experiment

Executing the unified pipeline across all four evaluation modes on the selected benchmark:
1. **Baseline FTM:** Fixed global threshold ($\theta = 0.80$)
2. **Improvement 1:** + Semantic Type-Aware Matching
3. **Improvement 2:** + Adaptive Threshold Optimization
4. **Improvement 3 / Final:** + Automated Conflict Resolution

In [ ]:
# Execute Four-Way Pipeline
all_modes = ["baseline", "improvement-1", "improvement-2", "final"]
all_metrics = {}
all_results = {}

for mode in all_modes:
    print(f"Executing Mode: {mode.upper()} ...")
    res = run_pipeline(
        dataset_name=DATASET_NAME,
        kg1_path=KG1_PATH,
        kg2_path=KG2_PATH,
        candidates_path=CANDIDATES_PATH,
        gold_path=GOLD_PATH,
        mode=mode,
        output_dir=OUTPUT_ROOT,
        use_gpu=USE_GPU,
        verbose=False
    )
    all_metrics[mode] = res["metrics"]
    all_results[mode] = res
    met = res["metrics"]
    prec_val = f"{met['precision']:.4f}" if met['precision'] is not None else "N/A"
    rec_val = f"{met['recall']:.4f}" if met['recall'] is not None else "N/A"
    f1_val = f"{met['f1']:.4f}" if met['f1'] is not None else "N/A"
    print(f"  -> [{mode:15}] Precision: {prec_val} | Recall: {rec_val} | F1: {f1_val} | Runtime: {met['runtime_seconds']}s")

# Improvement 3 is the complete Final pipeline
all_metrics["improvement-3"] = all_metrics["final"]

# Construct Master 4-Way Comparison DataFrame
comparison_df = build_comparison_dataframe(
    baseline_metrics=all_metrics["baseline"],
    imp1_metrics=all_metrics["improvement-1"],
    imp2_metrics=all_metrics["improvement-2"],
    imp3_metrics=all_metrics["final"]
)

## Section 6 — Comparative Metrics & Research Analysis

Below is the official 4-way comparison table synthesizing all metrics and differential gains:

In [ ]:
print("=" * 85)
print(f"MASTER 4-WAY COMPARISON BENCHMARK: {config['dataset_name'].upper()}")
print("=" * 85)
display(comparison_df)

# Extract key findings
base_f1 = all_metrics["baseline"]["f1"]
final_f1 = all_metrics["final"]["f1"]
if base_f1 is not None and final_f1 is not None:
    gain = final_f1 - base_f1
    print(f"\nKey Research Findings:")
    print(f"* Baseline F1:       {base_f1:.4f}")
    print(f"* Improvement 1 F1:  {all_metrics['improvement-1']['f1']:.4f}")
    print(f"* Improvement 2 F1:  {all_metrics['improvement-2']['f1']:.4f}")
    print(f"* Final Pipeline F1: {final_f1:.4f} (Net change vs Baseline: {gain:+.4f})")

## Section 7 — Deep Dive / Explainability for Each Improvement

Inspect the exact decision features generated by each architectural improvement.

In [ ]:
# 7.1 Inspect Improvement 1: Semantic Type Features
imp1_df = all_results["improvement-1"]["decisions"]
sem_cols = [c for c in ['entity_kg1', 'entity_kg2', 'similarity', 'semantic_type_kg1', 'semantic_type_kg2', 
                        'semantic_type_similarity', 'semantic_type_match', 'semantic_type_bonus', 'decision'] 
            if c in imp1_df.columns]
print("--- Improvement 1: Semantic Type Compatibility Sample ---")
display(imp1_df[sem_cols].head(5))

# 7.2 Inspect Improvement 2: Adaptive Threshold Breakdown
imp2_df = all_results["improvement-2"]["decisions"]
adapt_cols = [c for c in ['entity_kg1', 'entity_kg2', 'similarity', 'candidate_margin', 
                          'uncertainty_penalty', 'confidence_bonus', 'semantic_type_bonus', 
                          'adaptive_threshold', 'decision'] if c in imp2_df.columns]
print("\n--- Improvement 2: Adaptive Threshold Breakdown Sample ---")
display(imp2_df[adapt_cols].head(5))

# 7.3 Inspect Improvement 3: Conflict Resolution Log
conf_df = all_results["final"]["conflicts"]
print("\n--- Improvement 3: Automated Conflict Resolution Audit Log ---")
if conf_df is not None and not conf_df.empty:
    print(f"Total conflicts detected & resolved: {len(conf_df)}")
    display(conf_df.head(5))
else:
    print("No matching conflicts encountered in this candidate subset.")

## Section 8 — Explicit Knowledge Graph Pair Demonstration

Demonstrating that the pipeline can run on **arbitrary user-specified KG files** without requiring a registered dataset name (`DATASET_NAME = None`).

In [ ]:
# Execute pipeline using EXPLICIT Knowledge Graph files
print("Testing Explicit Knowledge Graph execution without dataset name...")

explicit_result = run_pipeline(
    dataset_name=None,
    kg1_path="dataset/movies/small_movie_kg1.ttl",
    kg2_path="dataset/movies/small_movie_kg2.ttl",
    candidates_path="dataset/movies/movie_candidates.csv",
    gold_path="dataset/movies/small_movie_gold.csv",
    mode="final",
    output_dir="results/custom_explicit_run",
    use_gpu=USE_GPU,
    verbose=True
)

print(f"Explicit KG Pair Matching Finished Successfully.")
print(f"Final Accepted Matches: {len(explicit_result['accepted_matches'])}")
print(f"Precision: {explicit_result['metrics']['precision']:.4f}")
print(f"Recall:    {explicit_result['metrics']['recall']:.4f}")
print(f"F1-Score:  {explicit_result['metrics']['f1']:.4f}")

## Section 9 — Summary & Verification

Summary of results and verification that all experiment artifacts have been saved reproducibly.

In [ ]:
# Print final run summary
print("=" * 65)
print("FTM EXPERIMENT SUMMARY & ARTIFACT AUDIT")
print("=" * 65)
print(f"Dataset:          {config.get('dataset_name', 'CUSTOM').upper()}")
print(f"KG1:              {config.get('kg1')}")
print(f"KG2:              {config.get('kg2')}")
if all_metrics["baseline"]["f1"] is not None:
    print(f"Baseline F1:      {all_metrics['baseline']['f1']:.4f}")
    print(f"Improvement 1 F1: {all_metrics['improvement-1']['f1']:.4f}")
    print(f"Improvement 2 F1: {all_metrics['improvement-2']['f1']:.4f}")
    print(f"Final F1:         {all_metrics['final']['f1']:.4f}")

target_results_dir = os.path.join(project_root, OUTPUT_ROOT, config.get('dataset_name', 'custom').lower())
print(f"\nSaved artifacts in {target_results_dir}:")
if os.path.exists(target_results_dir):
    for root_dir, dirs, files in os.walk(target_results_dir):
        rel = os.path.relpath(root_dir, project_root)
        for f in files:
            if f.endswith((".csv", ".json")):
                print(f"  - {os.path.join(rel, f)}")
print("=" * 65)